In [ ]:
# Cell 1 — Install
!pip uninstall unsloth unsloth_zoo torchao trl transformers bitsandbytes -y
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.1.1 datasets seqeval tqdm

In [1]:
# Cell 2 — Imports
import json
import random
import torch
from tqdm import tqdm
from datasets import load_dataset
from seqeval.metrics import f1_score, classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

random.seed(42)
torch.manual_seed(42)
print('Imports OK')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Free GPU: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB')

2026-05-15 17:12:46.642501: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778865166.666189     157 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778865166.674012     157 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778865166.694645     157 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778865166.694672     157 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778865166.694675     157 computation_placer.cc:177] computation placer alr

Imports OK
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Free GPU: 15.6 GB


In [2]:
# Cell 3 — Load Sayfullina
ds = load_dataset('jjzha/sayfullina')
print('Train:', len(ds['train']))
print('Test: ', len(ds['test']))
print('Sample:', ds['test'][0])

Train: 3705
Test:  1851
Sample: {'idx': 0, 'tokens': ['be', 'look', 'for', 'a', 'temporary', 'opportunity', 'within', 'a', 'progressive', 'and', 'dynamic', 'environment', ',', 'please', 'forward', 'your', 'cv', 'today'], 'tags_skill': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [3]:
# Cell 4 — BIO helpers
def get_tokens_and_tags(example):
    return example['tokens'], example['tags_skill']


def bio_from_bracket(tokens, output_text):
    tags = ['O'] * len(tokens)
    spans = []
    text = output_text
    while '@@' in text and '##' in text:
        start = text.find('@@')
        end   = text.find('##', start)
        if start == -1 or end == -1:
            break
        span = text[start+2:end].strip()
        spans.append(span)
        text = text[end+2:]
    for span in spans:
        span_tokens = span.split()
        n = len(span_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i : i + n] == span_tokens:
                if all(tags[i + j] == 'O' for j in range(n)):
                    tags[i] = 'B-SOFT'
                    for j in range(1, n):
                        tags[i + j] = 'I-SOFT'
                break
    return tags

print('BIO helpers defined')

BIO helpers defined


In [4]:
# Cell 5 — Dataset-specific ICL prompt (Sayfullina)
# Sayfullina characteristics:
#   - Soft skills only (interpersonal, behavioural)
#   - Short dense sentences, often grammatically incomplete
#   - Single or two-token spans most common
#   - Pre-tokenised (space-separated)

SYSTEM_ICL = (
    'You are a soft-skill extractor specialising in job advertisement sentences. '
    'Your task is to identify ONLY soft skills — interpersonal, behavioural, and '
    'personal competencies such as communication, leadership, teamwork, adaptability. '
    'Do NOT tag hard skills, tools, technologies, qualifications, or job titles. '
    'Sentences are pre-tokenised and may be grammatically incomplete — extract spans '
    'exactly as they appear, token-by-token. '
    'Wrap each soft skill span with @@...## brackets. '
    'Copy the sentence exactly with no other changes. '
    'If there are no soft skills, return the sentence unchanged. '
    'Output only the annotated sentence — no explanation, no commentary.'
)

def format_prompt_icl(sentence):
    return (
        f'<|im_start|>system\n{SYSTEM_ICL}<|im_end|>\n'
        f'<|im_start|>user\n{sentence}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )

# Sanity check
sample_tokens, _ = get_tokens_and_tags(ds['test'][0])
print('Sample prompt:')
print(format_prompt_icl(' '.join(sample_tokens)))

Sample prompt:
<|im_start|>system
You are a soft-skill extractor specialising in job advertisement sentences. Your task is to identify ONLY soft skills — interpersonal, behavioural, and personal competencies such as communication, leadership, teamwork, adaptability. Do NOT tag hard skills, tools, technologies, qualifications, or job titles. Sentences are pre-tokenised and may be grammatically incomplete — extract spans exactly as they appear, token-by-token. Wrap each soft skill span with @@...## brackets. Copy the sentence exactly with no other changes. If there are no soft skills, return the sentence unchanged. Output only the annotated sentence — no explanation, no commentary.<|im_end|>
<|im_start|>user
be look for a temporary opportunity within a progressive and dynamic environment , please forward your cv today<|im_end|>
<|im_start|>assistant



In [6]:
import os

def find_adapter(root='/kaggle/input'):
    for dirpath, dirnames, filenames in os.walk(root):
        if 'adapter_config.json' in filenames:
            print(dirpath)

find_adapter()


/kaggle/input/notebooks/acme105/p-4-sft-only-skill-extraction-from-job/checkpoints/checkpoint-462
/kaggle/input/notebooks/acme105/p-4-sft-only-skill-extraction-from-job/checkpoints/checkpoint-231
/kaggle/input/notebooks/acme105/p-4-sft-only-skill-extraction-from-job/checkpoints/checkpoint-693
/kaggle/input/notebooks/acme105/p-4-sft-only-skill-extraction-from-job/lora_adapter


In [7]:
# Cell 6 — Load base model + LoRA adapter from Phase 4
import os

ADAPTER_PATH = '/kaggle/input/notebooks/acme105/p-4-sft-only-skill-extraction-from-job/lora_adapter'
BASE_MODEL   = 'Qwen/Qwen2.5-3B-Instruct'

print('Adapter files:', os.listdir(ADAPTER_PATH))

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={'': 0},
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print(f'Model loaded. GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

Adapter files: ['adapter_model.safetensors', 'merges.txt', 'adapter_config.json', 'README.md', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'special_tokens_map.json', 'added_tokens.json']


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. GPU memory used: 6.4 GB


In [8]:
# Cell 7 — Sanity check: print 3 raw model outputs
for i in range(3):
    tokens, gold = get_tokens_and_tags(ds['test'][i])
    prompt  = format_prompt_icl(' '.join(tokens))
    inputs  = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 128,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Input:  {" ".join(tokens)}')
    print(f'Gold:   {gold}')
    print(f'Output: {decoded}')
    print(f'Pred:   {bio_from_bracket(tokens, decoded)}')
    print()

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Input:  be look for a temporary opportunity within a progressive and dynamic environment , please forward your cv today
Gold:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Output: be look for a temporary opportunity within a progressive and @@dynamic## environment , please forward your cv today
Pred:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Input:  effective member of the operation leadership team customer interaction to develop positive relationship a
Gold:   ['O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'I-SOFT', 'I-SOFT', 'O', 'O', 'O', 'O', 'O']
Output: effective member of the operation leadership team @@customer interaction## to develop positive relationship a
Pred:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-SOFT', 'I-SOFT', 'O', 'O', 'O', 'O', 'O']

Input:  linux shell be able to demonstrate a systematic and analytical approach to
Gold:   ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-S

In [9]:
# Cell 8 — Evaluate with ICL prompt
MAX_EXAMPLES = 200

test_examples = list(ds['test'])
random.shuffle(test_examples)
test_examples = test_examples[:MAX_EXAMPLES]

true_tags = []
pred_tags = []

for example in tqdm(test_examples, desc='Evaluating'):
    tokens, gold = get_tokens_and_tags(example)
    prompt       = format_prompt_icl(' '.join(tokens))

    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512,
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 128,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    )
    pred = bio_from_bracket(tokens, decoded)

    true_tags.append(gold)
    pred_tags.append(pred)

print('\nDone.')

Evaluating: 100%|██████████| 200/200 [05:32<00:00,  1.66s/it]


Done.


In [10]:
# Cell 9 — Results
print('=== Sayfullina — Qwen2.5-3B SFT + ICL (Phase 5) ===')
print(f'Examples evaluated: {len(true_tags)}')
print()
print(classification_report(true_tags, pred_tags))
f1 = f1_score(true_tags, pred_tags)
print(f'Strict F1: {f1:.4f}')
print()
print('Phase 1 (random 5-shot, no RAG):        0.1432 F1')
print('Phase 2 (RAG-1 + RAG-2, no verifier):   0.3154 F1')
print('Phase 3 (+ Verifier):                   0.3429 F1')
print('Phase 4 (SFT only):                     0.8150 F1')
print('Paper SRICL target (full pipeline):      0.6118 F1')

=== Sayfullina — Qwen2.5-3B SFT + ICL (Phase 5) ===
Examples evaluated: 200

              precision    recall  f1-score   support

        SOFT       0.77      0.74      0.76       201

   micro avg       0.77      0.74      0.76       201
   macro avg       0.77      0.74      0.76       201
weighted avg       0.77      0.74      0.76       201

Strict F1: 0.7551

Phase 1 (random 5-shot, no RAG):        0.1432 F1
Phase 2 (RAG-1 + RAG-2, no verifier):   0.3154 F1
Phase 3 (+ Verifier):                   0.3429 F1
Phase 4 (SFT only):                     0.8150 F1
Paper SRICL target (full pipeline):      0.6118 F1
